# Seurat to AnnData

Python-first notebook entry point for converting the Shi paper-QC, Varela DIV30, and Varela DIV90 Seurat objects into cached AnnData `.h5ad` files.

Run this with the `Python (mge-organoid-python)` kernel. Set `PROJECT_ROOT` before launching Jupyter:

```bash
export PROJECT_ROOT=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder
```

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
if len(cwd.parents) >= 2:
    candidate_roots.append(cwd.parents[1])

for root in candidate_roots:
    src = root / "python_notebooks" / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        repo_root = root
        break
else:
    raise RuntimeError("Could not locate python_notebooks/src from the current notebook directory")

repo_root

In [ ]:
from mge_organoid_python import (
    SeuratToAnnDataConverter,
    default_studies,
    resolve_project_root,
    validate_source_paths,
)

project_root = resolve_project_root()
studies = default_studies()

print(f"PROJECT_ROOT = {project_root}")
for study in studies:
    print(f"{study.study_id}: {study.seurat_path}")

In [ ]:
missing = validate_source_paths(studies)
if missing:
    for study_id, path in missing:
        print(f"MISSING {study_id}: {path}")
    raise FileNotFoundError("One or more canonical Seurat inputs are missing")

print("All canonical Seurat inputs exist.")

In [ ]:
converter = SeuratToAnnDataConverter(project_root=project_root)
print(f"AnnData cache directory: {converter.output_dir}")

for study in studies:
    print(study.study_id, "->", converter.output_path(study), "needs_conversion=", converter.needs_conversion(study))

The next cell performs conversion. For large Seurat objects, run it in an interactive compute or GUI session rather than using login-node resources for long jobs.

In [ ]:
import pandas as pd

adatas, reports = converter.convert_many(studies)
reports_df = pd.DataFrame([report.as_dict() for report in reports])
reports_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(studies), figsize=(5 * len(studies), 4), constrained_layout=True)
if len(studies) == 1:
    axes = [axes]

for ax, study in zip(axes, studies):
    adata = adatas[study.study_id]
    umap = adata.obsm["X_umap"]
    ax.scatter(umap[:, 0], umap[:, 1], s=1, linewidths=0, alpha=0.6)
    ax.set_title(f"{study.label}\nn={adata.n_obs:,}")
    ax.set_xlabel("UMAP_1")
    ax.set_ylabel("UMAP_2")

plt.show()

In [ ]:
# Access individual AnnData objects by study id.
shi = adatas["shi_2019_paper_qc"]
varela_div30 = adatas["varela_div30"]
varela_div90 = adatas["varela_div90"]

shi, varela_div30, varela_div90